# Portfolio


Strategy: **sentiment_asynchronous**.

## Settings

- **Purpose:** Define the single meta-filtered account.
- **Settings:** `initial_aum=100000 USD`; `K=10` per side; `step_size=0.10`; fees/slippage = 1 bp each per side; score half-life = 390 session minutes.
- **Data:** Shared immutable defaults and frozen price calibration.
- **Findings:**
- **Decision:** The final statistics notebook performs the full simulation once; this notebook writes no ledger.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.preprocessing.market_data import ResearchPaths
from src.backtesting.portfolio_management import PortfolioSettings
PROJECT_ROOT = Path.cwd().resolve().parents[2]
PERIOD = "2025-02-01_2025-12-31"
HORIZON_BARS = 1000
K = 10
STEP_SIZE = 0.10
INITIAL_AUM = 100_000.0
BROKER_FEE_BPS = 1.0
SLIPPAGE_BPS = 1.0
HALF_LIFE_MINUTES = 390.0
paths = ResearchPaths(PROJECT_ROOT, model_kind="sentiment", period=PERIOD)
settings = PortfolioSettings(
    k=K, step_size=STEP_SIZE, initial_aum=INITIAL_AUM,
    broker_fee_bps=BROKER_FEE_BPS, slippage_bps=SLIPPAGE_BPS,
    half_life_minutes=HALF_LIFE_MINUTES,
)
from src.backtesting.portfolio_management import load_prediction_events
events = load_prediction_events(paths, horizon_bars=HORIZON_BARS)
display(pd.Series(vars(settings), name="value").to_frame())

## Candidates and Weights

- **Purpose:** Inspect within-symbol probability targets at the first development timestamp.
- **Settings:** `K=10`; `step_size=0.10`; per-symbol weight cap = 5% of AUM; pass signals count as zero.
- **Data:** At the first development timestamp, only newly arriving events are active.
- **Findings:**
- **Decision:** Keep the snapshot descriptive; the account simulator applies candidate ranking and eligibility rules.

In [ ]:
development = events.loc[events.partition.eq("development")]
example_time = development.event_start.min()
example_active = development.loc[development.event_start.eq(example_time)].copy()
from src.backtesting.bet_sizing import average_symbol_targets
display(average_symbol_targets(example_active, settings.k, settings.step_size).to_frame())

## Queues and Replacement

- **Purpose:** Document the causal account transition rules.
- **Settings:** Sentiment tails = 20% each, at least `K=10` members; positive score; 390-session-minute half-life; incumbent wins ties; exit request 1 minute before final close.
- **Data:** Runtime active events, executable quotes, and separate long/short queues.
- **Findings:**
- **Decision:** Observed event expiry changes only that security’s average. Replacement requires executable quotes for both legs after the decision; no speculative incumbent sale.

In [ ]:
rules = pd.Series({
    "sizing": "transform each event; average within symbol including zero pass signals; discretize",
    "new_or_increased_position": "tail eligibility, aligned latest direction, take, positive score, limit price",
    "reduction": "no entry limit; execute after the decision",
    "replacement": "strictly higher score; both legs executable in the same quote batch",
    "final_liquidation": "request exits one regular-session minute before final close",
    "corporate_actions": "outside this simplified fixed-symbol evaluation",
    "saved_results": "strategy statistics, account curves, and comparison metadata",
}, name="policy")
display(rules.to_frame())